# El toyset — generación

Corre **el plan** en sus tres pasos y deja los resultados en disco. No dibuja
nada: las figuras y las tablas las arma `Toyset_Plot.ipynb`, que lee lo que este
cuaderno escribe.

1. **Entreno la fuente** con cross-entropy y nada más, parando en su mejor época
   de validación de la **fuente** — que no es fuga: `y_s` lo tenemos.
2. **Transfiero** los pesos del encoder de la fuente al encoder del blanco.
3. **Meto fuente + blanco** y dejo mover el encoder y la cabeza de los dos, con
   el **regularizador de Rényi fuente↔blanco** prendido, normalizando las `K`
   sobre `tr(K)` incluida la mixta, y sin criterio de parada: la fase 2 corre
   sus pasos hasta el final.

Y después el paso 4: los tres pasos **con y sin normalización**, probando la de
toda la vida (BatchNorm) y la que normaliza por features (LayerNorm).

**La fase 1 no lleva ningún término de latente.** Llevó tres a lo largo del
estudio — la Hellinger global contra `N(0,I)`, el par de entropías de Rényi sobre
las clases, y la KL — y los tres se retiraron. Medido sobre 18 celdas: cambiar el
término abre 0,1322 de acierto en el blanco y cambiar la normalización abre
0,2438. Lo que decide es la arquitectura, no el término.

**La red es de una sola capa**, a propósito: `entrada → Linear → ReLU → [BatchNorm
| LayerNorm | nada]`. 64 a 96 parámetros contra los 1.712 de la red completa. La
pregunta del experimento es si los kernels le dan al modelo suficiente
información para no depender de la arquitectura.

## Dónde está el repositorio

In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()

## El corrimiento de dominio, en dos partes

Ninguno de los dos cuadernos lo decía, y es la decisión central del
experimento. El corrimiento tiene dos partes con papeles distintos:

**1. Una rotación que preserva Σ exactamente.**

    A = Σ_s^½ · R(θ) · Σ_t^-½        cumple    A Σ_t Aᵀ = Σ_s

Centrada en la media y recentrada a `μ_s`, el blanco queda con la media y la
covarianza **enteras** de la fuente. Medido: `1e-15` sin ruido, `1e-3` con el
ruido al 2,5% — unas **30 veces por debajo** del piso de dos sorteos
independientes de la misma distribución (`3e-2` a `4.6e-2`). AdaBN corrige media
y varianza por canal; CORAL, la covarianza entera. Sobre esta parte los dos son
no-ops **demostrables**, no empíricamente débiles.

Antes era una rotación genérica de 30°, y ése era el problema: una rotación
genérica **mueve** la covarianza (0.269 relativo), así que AdaBN arreglaba parte
de ella y lo que CREDA ganaba quedaba confundido con exactamente lo que AdaBN
arregla.

**2. Una traslación de 3σ, que AdaBN sí cierra.** No está de adorno. Con la
rotación sola, `.predict` queda en el azar, y entonces el ruteo por clase de
CREDA —que sale de `softmax(cab(f_t))`— es ruido y CREDA no tiene de dónde
arrancar. Medido en ese régimen: AdaBN 0.4675 contra un `.predict` de 0.4675
(no-op exacto) y CREDA entre −0.006 y −0.046. Con ruteo de **oráculo** en cambio
sube +0.058 / +0.163 / +0.130 — el kernel **tiene** la información, lo que falta
es el arranque.

**Por eso la atribución limpia es CREDA menos AdaBN**, no CREDA menos
`.predict`: AdaBN cierra lo que el segundo orden puede cerrar, y lo que quede
encima tiene brecha de covarianza cero.

**El ángulo está en 10° por medición**, no por gusto. Con la traslación puesta,
CREDA − AdaBN sobre `rollo` da +0.105 (0°), **+0.133 (10°)**, +0.089 (20°),
+0.062 (30°), +0.031 (45°), −0.009 (60°), y AdaBN cae monótono 0.765 → 0.594 —
que es la confirmación medida de que la rotación no es corregible por segundo
orden. Sobre 3 semillas la ventana 10–20° es sólida; 10° contra 20° no es
resoluble y haría falta re-medir con 5.

Se cambia con `ANGULO_SIGMA` en `src/CREDA/toy_experiment.py`, y sale impreso en
`banderas.json`.

## El montaje y las banderas de esta corrida

In [2]:
import sys
sys.path.insert(0, str(ROOT/'src'))

import numpy as np

from CREDA.toy_experiment import (
    FAMILIAS, POS_FILAS, RHOS, SEMILLAS_BUSQUEDA, SEMILLAS_REPORTE, banderas,
    barrer_animaciones, barrer_contaminacion, barrer_gradiente, barrer_tablas,
    buscar_hp1, buscar_hp2, guardar, guardar_contaminacion,
    MODOS_CONTAMINACION, fusionar_contaminacion,
)

# Donde van los resultados. El cuaderno de graficas lee de aca y no vuelve a
# entrenar: eso es toda la razon de que el cuaderno este partido en dos.
SALIDA = ROOT/'CREDA/Results/tesis/toyset'
print('resultados ->', SALIDA.relative_to(ROOT))
for k, v in banderas().items():
    print(f'  {k:<22} {v}')

resultados -> CREDA/Results/tesis/toyset
  UNA_CAPA               True
  DIM                    16
  CLASES                 3
  LOTE                   128
  SEMILLA                0
  EPOCAS_1               200
  PASOS_2                200
  LR_1                   0.01
  LR_2                   0.001
  SIGMA_DE               train completo de la fuente, congelado tras la fase 1
  NORMALIZAR_POR_COTAS   True
  FORMA_DE_COTAS         entropy-cosh
  BALANCE_DE_MASA        trace
  CUADROS                21
  POS_FILAS              ['ninguna', 'batchnorm', 'layernorm']
  SEMILLAS_BUSQUEDA      [100, 101, 102]
  SEMILLAS_REPORTE       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
  TRIALS_1               25
  TRIALS_2               25
  TRASLADO_REL           3.0
  ANGULO_SIGMA_GRADOS    10.0
  FRAC_TRAIN             0.6
  FRAC_VALID             0.2
  PARADA_TEMPRANA        ninguna: las dos fases corren su techo completo y el criterio elige la mejor epoca de TODA la corrida
  PARADA_TEMPRANA_CAMBIO 2026-10-

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## El barrido

Esto es lo que tarda. Las dos fases por familia, por normalización y por régimen
de la fuente, más cinco semillas para las tablas.

In [3]:
import time
_t0 = time.time()

# ---------------------------------------------------------------------------
# 1. Los HP de la FASE 1, uno por toyset.
#
# Una config por familia, COMPARTIDA por las tres normalizaciones: si cada brazo
# tuviera sus propios HP, la comparacion entre normalizaciones quedaria
# confundida con los HP y el eje del experimento se arruina.
#
# La arquitectura NO se busca --- es el eje, y elegirla por el acierto de la
# fuente cuesta 0.13-0.17 en el blanco (medido). El objetivo es el acierto de
# VALID de la fuente, y como casi todo empata, los empates los rompe el CE.
# ---------------------------------------------------------------------------
print(f'busqueda de fase 1 · semillas {SEMILLAS_BUSQUEDA}')
HP1, R1 = {}, {}
for _f in FAMILIAS:
    HP1[_f['nombre']], R1[_f['nombre']] = buscar_hp1(_f)
print(f'  fase 1 lista · {time.time()-_t0:.0f}s')

# ---------------------------------------------------------------------------
# 2. Los HP de la FASE 2, con la fase 1 congelada y entrenada UNA vez por
#    (normalizacion, semilla). Asi todos los trials comparan sobre el mismo
#    modelo de fase 1 y la diferencia entre ellos es apareada.
#
#    El objetivo es el MINIMO DEL VIGIA --- el encoder del blanco sobre la
#    validacion de la FUENTE, con etiquetas de la fuente. Cero y_t, asi que el
#    mismo criterio corre sobre datos que no las tienen.
#
#    Medido sobre 75 trials: el vigia ordena mucho mejor que el Renyi de valid
#    (Pearson -0.984 contra -0.129 en toroide), y el HP que elige da 0.6584 de
#    media contra 0.5749, con el oraculo en 0.6856. Es la misma cantidad que la
#    senal de parada, y eso es coherente: las dos preguntas son 'cuanto se
#    adapto', una en el tiempo y otra entre configuraciones.
#
#    La ELECCION entre los dos escalares se hizo con una comparacion contra y_t,
#    una vez. La OPERACION no lo usa en ningun paso.
# ---------------------------------------------------------------------------
print(f'{chr(10)}busqueda de fase 2 · semillas {SEMILLAS_BUSQUEDA}')
HP2, R2 = {}, {}
for _f in FAMILIAS:
    HP2[_f['nombre']], R2[_f['nombre']] = buscar_hp2(_f, HP1[_f['nombre']])
print(f'  fase 2 lista · {time.time()-_t0:.0f}s')

HP = {n: (HP1[n], HP2[n]) for n in HP1}
print(f'{chr(10)}los HP que quedaron, por toyset:')
for _n in HP1:
    print(f'  {_n}')
    print('    fase 1  ' + '  '.join(f'{k}={v}' for k, v in HP1[_n].items()))
    print('    fase 2  ' + '  '.join(
        f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
        for k, v in HP2[_n].items()))

# ---------------------------------------------------------------------------
# 3. Los barridos con los mejores HP. Las semillas del REPORTE son disjuntas de
#    las de busqueda: si se buscara y se reportara en las mismas, los HP
#    quedarian ajustados a las semillas que despues se publican.
# ---------------------------------------------------------------------------
print(f'{chr(10)}barridos finales · semillas de reporte {SEMILLAS_REPORTE}')
POR, DAT = barrer_animaciones(HP=HP)
print(f'  animaciones: {len(POR)} corridas · {time.time()-_t0:.0f}s')

CELDAS, ADA = barrer_tablas(HP=HP, semillas=SEMILLAS_REPORTE)
print(f'  tablas: {len(CELDAS)} celdas × {len(SEMILLAS_REPORTE)} semillas '
      f'· {time.time()-_t0:.0f}s')

GRAD = barrer_gradiente(HP=HP)
print(f'  gradiente: {len(GRAD)} corridas de diagnóstico · {time.time()-_t0:.0f}s')

escritos = guardar(SALIDA, POR, DAT, CELDAS, ADA, GRAD, HP1, HP2, R2)
for _p in escritos:
    print(f'  {_p.relative_to(ROOT)}  {_p.stat().st_size/1e6:.2f} MB')
print(f'{chr(10)}listo · {time.time()-_t0:.0f}s')

busqueda de fase 1 · semillas [100, 101, 102]


    lunas    fase 1: 25 trials, acc 0.9904, 10 dentro de 1e-2 -> desempata el CE (0.0036)


    toroide  fase 1: 25 trials, acc 0.9991, 24 dentro de 1e-2 -> desempata el CE (0.0012)


    rollo    fase 1: 25 trials, acc 0.9997, 22 dentro de 1e-2 -> desempata el CE (0.0004)
  fase 1 lista · 285s

busqueda de fase 2 · semillas [100, 101, 102]


    lunas    fase 2: 25 trials, cef 2.6935 (el objetivo), vigía 0.7117, Rényi de valid 0.1161


...

  toroide   30 corridas + 10 AdaBN ·     74s acumulados


  rollo     30 corridas + 10 AdaBN ·    106s acumulados
  tablas: 9 celdas × 10 semillas · 918s


  lunas    3 corridas de diagnóstico


  toroide  3 corridas de diagnóstico


  rollo    3 corridas de diagnóstico
  gradiente: 9 corridas de diagnóstico · 929s
  CREDA/Results/tesis/toyset/banderas.json  0.00 MB
  CREDA/Results/tesis/toyset/barrido.json  0.42 MB
  CREDA/Results/tesis/toyset/lunas.npz  0.60 MB
  CREDA/Results/tesis/toyset/toroide.npz  0.59 MB
  CREDA/Results/tesis/toyset/rollo.npz  0.59 MB

listo · 929s


## El sopesado bajo contaminación

La pregunta de esta sección: **el sopesado por confianza (Ecs. 19 y 26) gana
algo cuando el daño al blanco es DESIGUAL.**

Bajo el balance por traza (Ec. 32) la escala COMÚN de `w` se cancela —
`K~_t = c² K_t` sobre `tr(K~_t) = c² tr(K_t)`, y `K~_st = c K_st` sobre
`c·√(…)` — así que una perturbación que degrade todos los puntos por igual
mide **cero por construcción**. Eso está medido y ahora también probado, en
`tests/test_toy_contamination.py`; por eso el eje de ruido uniforme **no se
corre** y es un test.

Lo que sobrevive a la cancelación es el descuento RELATIVO, y para verlo hace
falta que unos puntos estén dañados y otros no.

**El mecanismo.** Una fracción `ρ` de los puntos del **train del blanco** se
reubica en la región de otra clase conservando su etiqueta verdadera:
`x_j ← x_j − μ_{y_j} + μ_{c'}`. El **test del blanco no se toca**, así que el
acierto es comparable a lo largo de la escalera y mide el daño, no el
corrimiento del conjunto de evaluación.

**Escalera declarada**: `ρ = (0.00, 0.10, 0.20, 0.30)`, el cero incluido.

**Cuatro brazos por escenario**, en escalera de intervención: `.predict`,
`AdaBN`, `CREDA sin sopesado`, `CREDA con sopesado`. Los dos últimos difieren
**sólo** en `use_entropy_weighting`. La celda es BatchNorm y eso es derivado,
no elegido: AdaBN sólo existe donde hay buffers que reestimar.

**UNA búsqueda de fase 2 por escenario, corrida una sola vez y compartida por
los dos brazos de sopesado.** No por brazo —sería «el mejor de `w` prendido»
contra «el mejor de `w` apagado», otra afirmación— y no reusada del nivel
limpio —leería la curva en un punto sintonizado para condiciones que ya no
valen—. Los HP de la fase 1 se buscan una sola vez por familia, arriba: la
fase 1 no mira el blanco.

**Predicho por escrito, antes de correr.** `w_j = 1 − H₂(g_j)/ln C` mide
INCERTIDUMBRE, no error. Un punto contaminado que cae bien adentro de la clase
equivocada queda confiadamente equivocado, tiene `w` ALTO y **crece** en la
figura. Sólo los que caen entre clases se achican. Es el límite real del
sopesado, no un panel roto.

In [4]:
# Reusa los HP de la FASE 1 que la busqueda de arriba ya eligio: la fase 1 no
# mira el blanco y la perturbacion es del lado del blanco, asi que la fuente de
# un escenario es la misma que la del nivel limpio. La fase 2 SI se busca por
# escenario, adentro de `barrer_contaminacion`, una vez y compartida por los
# dos brazos de sopesado.
#
# La escalera y los trials van DECLARADOS en el modulo: aca no se pasan, para
# que lo que corra sea lo que `RHOS` y `TRIALS_2` dicen y no un numero escrito
# en un cuaderno.
#
# **LAS DOS perturbaciones, y fusionadas en UN artefacto.** Un barrido corre
# una; las dos predicen lo CONTRARIO sobre el par de medias al mismo rho
# --- `frontera` es el caso donde el sopesado PUEDE funcionar, `reubicado` el
# adversario donde se rompe demostrablemente --- asi que la seccion necesita las
# dos y ninguna de las dos es la conclusion. El padron sale de
# `MODOS_CONTAMINACION` y no de dos nombres escritos aca: una perturbacion
# nueva en el modulo entra a la corrida sola.
_tc = time.time()
print(f'escalera de contaminacion {RHOS} {chr(183)} '
      f'semillas de reporte {SEMILLAS_REPORTE} {chr(183)} '
      f'perturbaciones {list(MODOS_CONTAMINACION)}')
_barridos = []
for _modo in MODOS_CONTAMINACION:
    _tm = time.time()
    print(f'{chr(10)}--- perturbacion {_modo} ---')
    # `HP2=HP2`: la fase 2 NO se busca por escenario, se recibe ya elegida
    # del nivel limpio. Decisión del operador del 2026-10-18. Lo que ahorra,
    # medido: la búsqueda por escenario era el 76% del flujo entero --- dos
    # perturbaciones por tres familias por cuatro ρ por `TRIALS_2` trials ---
    # y los brazos que esta sección compara son seis minutos.
    #
    # Lo que cambia, dicho porque cambia lo que la celda afirma: los HP quedan
    # elegidos sobre el nivel limpio y aplicados FUERA DE MUESTRA a los
    # contaminados. El registro lo lleva por celda --- `busquedas = 0` y
    # `hp2_inyectado = True` --- así que una tabla no puede atribuirle a un
    # escenario una búsqueda que nadie corrió.
    _barridos.append(barrer_contaminacion(
        HP1=HP1, HP2=HP2, semillas=SEMILLAS_REPORTE, modo=_modo,
        al_paso=lambda *a: print(*a)))
    print(f'  {_modo}: {len(_barridos[-1]["celdas"])} escenarios '
          f'{chr(183)} {time.time()-_tm:.0f}s')

# UN artefacto con las dos adentro. La perturbacion va DENTRO de la clave de
# cada celda, asi que el lado que dibuja lee un solo archivo y elige el
# escenario por el par `(perturbacion, rho)` --- nunca por el rho solo, que
# elegiria una de las dos sin decir cual.
CONTAM = fusionar_contaminacion(*_barridos)
print(f'{chr(10)}  contaminacion: {len(CONTAM["celdas"])} escenarios '
      f'{chr(183)} perturbaciones {CONTAM["perturbaciones"]} '
      f'{chr(183)} {time.time()-_tc:.0f}s')

escritos_c = guardar_contaminacion(SALIDA, CONTAM)
for _p in escritos_c:
    print(f'  {_p.relative_to(ROOT)}  {_p.stat().st_size/1e6:.2f} MB')

# Los HP que cada escenario eligio, uno por fila: una tabla sin decir bajo que
# hiperparametros se midio no se puede leer, y aca hay uno por nivel y por
# perturbacion. Con dos barridos fusionados `hp2` queda anidado POR
# perturbacion, asi que la primera clave es la perturbacion y no la familia.
print(f'{chr(10)}los HP de fase 2 por escenario:')
for _p in CONTAM['perturbaciones']:
    for _n in sorted(CONTAM['hp2'][_p]):
        for _r, _h in CONTAM['hp2'][_p][_n].items():
            print(f'  {_p:10s} {_n:8s} rho={_r}  ' + '  '.join(
                f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
                for k, v in _h.items()))
print(f'{chr(10)}listo {chr(183)} {time.time()-_t0:.0f}s en total')


escalera de contaminacion (0.0, 0.1, 0.2, 0.3) · semillas de reporte [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] · perturbaciones ['reubicado', 'frontera']

--- perturbacion reubicado ---


  lunas · reubicado · rho=0.00 · fase 2 RECIBIDA, sin busqueda


    lunas · rho=0.00 listo ·     39s acumulados
  lunas · reubicado · rho=0.10 · fase 2 RECIBIDA, sin busqueda


    lunas · rho=0.10 listo ·     70s acumulados
  lunas · reubicado · rho=0.20 · fase 2 RECIBIDA, sin busqueda


    lunas · rho=0.20 listo ·    102s acumulados
  lunas · reubicado · rho=0.30 · fase 2 RECIBIDA, sin busqueda


...

    rollo · rho=0.00 listo ·    253s acumulados
  rollo · frontera · rho=0.10 · fase 2 RECIBIDA, sin busqueda


    rollo · rho=0.10 listo ·    277s acumulados
  rollo · frontera · rho=0.20 · fase 2 RECIBIDA, sin busqueda


    rollo · rho=0.20 listo ·    302s acumulados
  rollo · frontera · rho=0.30 · fase 2 RECIBIDA, sin busqueda


    rollo · rho=0.30 listo ·    326s acumulados
  frontera: 12 escenarios · 326s

  contaminacion: 24 escenarios · perturbaciones ['reubicado', 'frontera'] · 652s


  CREDA/Results/tesis/toyset/contaminacion.json  5.10 MB
  CREDA/Results/tesis/toyset/contaminacion_lunas.npz  3.55 MB
  CREDA/Results/tesis/toyset/contaminacion_rollo.npz  3.39 MB
  CREDA/Results/tesis/toyset/contaminacion_toroide.npz  3.34 MB

los HP de fase 2 por escenario:
  reubicado  lunas    rho=0.00  LR_2=0.003029  WD_2=1.56e-06  LOTE_2=128  Q_COMPUERTA=0.25  PRECISION_TECHO=1  RHO_EMB=0  RADIO_EMB=4  RHO_UNI=0  DESPL_UNI=0  CRUZADO_SOPESADO=True  RHO_T_SIN_GRAD=False  PASO_W=0  PASO_COND=0
  reubicado  lunas    rho=0.10  LR_2=0.003029  WD_2=1.56e-06  LOTE_2=128  Q_COMPUERTA=0.25  PRECISION_TECHO=1  RHO_EMB=0  RADIO_EMB=4  RHO_UNI=0  DESPL_UNI=0  CRUZADO_SOPESADO=True  RHO_T_SIN_GRAD=False  PASO_W=0  PASO_COND=0
  reubicado  lunas    rho=0.20  LR_2=0.003029  WD_2=1.56e-06  LOTE_2=128  Q_COMPUERTA=0.25  PRECISION_TECHO=1  RHO_EMB=0  RADIO_EMB=4  RHO_UNI=0  DESPL_UNI=0  CRUZADO_SOPESADO=True  RHO_T_SIN_GRAD=False  PASO_W=0  PASO_COND=0
  reubicado  lunas    rho=0.30  LR_2=0.00302